# Sensitivity and Stress Analysis

This notebook studies how a European call option responds to changes in
spot price, volatility, interest rates, and time to maturity.

It finishes with a small set of clearly defined stress scenarios.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.black_scholes import black_scholes_greeks, black_scholes_price

## Base option

In [ ]:
base = {
    "spot": 100.0,
    "strike": 100.0,
    "maturity": 1.0,
    "rate": 0.05,
    "volatility": 0.20,
    "dividend_yield": 0.00,
}

base_price = black_scholes_price(**base, option_type="call")
base_greeks = black_scholes_greeks(**base, option_type="call")

print("Base call price:", round(base_price, 4))
pd.Series(base_greeks, name="Value").round(6)

## Spot sensitivity

In [ ]:
spot_values = np.linspace(60, 140, 81)
spot_prices = [
    black_scholes_price(
        **{**base, "spot": spot},
        option_type="call",
    )
    for spot in spot_values
]

plt.figure(figsize=(7, 4))
plt.plot(spot_values, spot_prices)
plt.xlabel("Spot price")
plt.ylabel("Call price")
plt.title("Sensitivity to spot price")
plt.tight_layout()
plt.show()

## Volatility sensitivity

In [ ]:
volatility_values = np.linspace(0.05, 0.60, 56)
volatility_prices = [
    black_scholes_price(
        **{**base, "volatility": volatility},
        option_type="call",
    )
    for volatility in volatility_values
]

plt.figure(figsize=(7, 4))
plt.plot(volatility_values, volatility_prices)
plt.xlabel("Volatility")
plt.ylabel("Call price")
plt.title("Sensitivity to volatility")
plt.tight_layout()
plt.show()

## Interest-rate sensitivity

In [ ]:
rate_values = np.linspace(-0.02, 0.12, 71)
rate_prices = [
    black_scholes_price(
        **{**base, "rate": rate},
        option_type="call",
    )
    for rate in rate_values
]

plt.figure(figsize=(7, 4))
plt.plot(rate_values, rate_prices)
plt.xlabel("Continuously compounded rate")
plt.ylabel("Call price")
plt.title("Sensitivity to interest rates")
plt.tight_layout()
plt.show()

## Maturity sensitivity

In [ ]:
maturity_values = np.linspace(0.05, 3.0, 60)
maturity_prices = [
    black_scholes_price(
        **{**base, "maturity": maturity},
        option_type="call",
    )
    for maturity in maturity_values
]

plt.figure(figsize=(7, 4))
plt.plot(maturity_values, maturity_prices)
plt.xlabel("Time to maturity in years")
plt.ylabel("Call price")
plt.title("Sensitivity to maturity")
plt.tight_layout()
plt.show()

## Stress scenarios

In [ ]:
scenarios = {
    "Base": {},
    "Spot down 20%": {"spot": base["spot"] * 0.80},
    "Volatility up 25%": {
        "volatility": base["volatility"] * 1.25
    },
    "Rates up 200 bps": {"rate": base["rate"] + 0.02},
    "Spot down 20% and volatility up 50%": {
        "spot": base["spot"] * 0.80,
        "volatility": base["volatility"] * 1.50,
    },
    "Spot up 20% and volatility down 25%": {
        "spot": base["spot"] * 1.20,
        "volatility": base["volatility"] * 0.75,
    },
}

stress_rows = []
for name, changes in scenarios.items():
    stressed = {**base, **changes}
    price = black_scholes_price(**stressed, option_type="call")
    stress_rows.append(
        {
            "Scenario": name,
            "Option price": price,
            "Price change": price - base_price,
            "Percentage change": 100 * (price / base_price - 1),
        }
    )

stress_results = pd.DataFrame(stress_rows)
stress_results.round(4)

## Interpretation

- A call option generally gains value as the underlying price rises.
- Higher volatility generally increases the value of both calls and puts.
- Rate sensitivity depends on the option type and contract assumptions.
- Combined stresses can behave differently from the sum of individual effects because option values are nonlinear.
- These results are conditional on Black–Scholes assumptions, including constant volatility and rates, continuous trading, and lognormal asset prices.